# Claude Managed Agents로 Slack 데이터 분석 봇 만들기

## 들어가며

[`data_analyst_agent.ipynb`](data_analyst_agent.ipynb)의 에이전트를, Slack 공식 앱 개발 프레임워크인 [Bolt for Python](https://docs.slack.dev/tools/bolt-python/)으로 만든 Slack 봇으로 감싸 보겠습니다. 질문과 CSV 첨부 파일을 달아 봇을 멘션하면 스레드에 서술형 보고서가 올라옵니다. 후속 메시지는 같은 세션에서 이어집니다.

```text
user: @databot what's driving Q1 revenue?  [sales.csv]
    │
    ▼
bot uploads the CSV and starts an agent session
    │
    ▼
bot streams the agent's progress back to the thread
    │
    ▼
bot posts the finished report to the thread
```

### 배울 내용

- Slack 멘션으로 에이전트 실행 시작하기
- 에이전트의 진행 상황을 스레드 업데이트로 보여 주기
- 완성된 보고서를 스레드에 게시하기
- 후속 답글로 대화 이어 가기

### 사전 준비

1. 아래 설치 셀을 실행하세요.

2. [Slack 앱](https://api.slack.com/apps)을 만드세요. **Create New App → From a manifest**를 선택하고 [`slack_app_manifest.yaml`](example_data/slack_data_bot/slack_app_manifest.yaml)을 붙여 넣은 뒤 워크스페이스에 설치합니다. 이 매니페스트는 소켓 모드(Slack이 WebSocket으로 이벤트를 전달하므로 공개 URL이 필요 없습니다)와 필요한 스코프를 활성화합니다. 그런 다음 토큰 두 개를 확보하세요.

   - **OAuth & Permissions** → Bot User OAuth Token(`xoxb-...`) 복사
   - **Basic Information → App-Level Tokens** → `connections:write` 스코프로 하나 생성(`xapp-...`)

   봇을 넣고 싶은 채널에서 `/invite @databot`을 실행하세요.

3. [`data_analyst_agent.ipynb`](data_analyst_agent.ipynb)를 실행하세요. `ANALYST_ENV_ID`, `ANALYST_AGENT_ID`, `ANALYST_AGENT_VERSION`을 `.env`에 저장합니다.

아래 설정 셀은 Slack 토큰을 입력받아 `.env`에 저장하므로 재시작할 때 다시 입력하지 않아도 됩니다(미리 `.env`에 넣어 두면 입력 절차를 건너뜁니다). `.env`는 이미 `.gitignore`에 있습니다. 절대 버전 관리에 커밋하지 마세요. Slack 워크스페이스가 없더라도 코드를 읽어 볼 수는 있습니다. 각 절에서 무엇을 하는지 설명합니다. 다만 봇을 실행하려면 워크스페이스가 필요합니다.

In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv slack_bolt requests markdown-to-mrkdwn

In [ ]:
import io
import os
import threading
from getpass import getpass

import requests
from anthropic import Anthropic
from dotenv import load_dotenv, set_key
from markdown_to_mrkdwn import SlackMarkdownConverter
from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler

load_dotenv(override=True)

# Prompt for Slack tokens on first run and save them to .env.
for key in ("SLACK_BOT_TOKEN", "SLACK_APP_TOKEN"):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")
        set_key(".env", key, os.environ[key])

client = Anthropic()
app = App(token=os.environ["SLACK_BOT_TOKEN"])

for key in ("ANALYST_ENV_ID", "ANALYST_AGENT_ID", "ANALYST_AGENT_VERSION"):
    if not os.environ.get(key):
        raise RuntimeError(f"{key} not set. Run data_analyst_agent.ipynb first.")

# Set these from the IDs saved by the data analyst notebook. Reusing
# the agent and environment avoids re-provisioning on every bot restart.
ANALYST_AGENT = {
    "id": os.environ["ANALYST_AGENT_ID"],
    "version": int(os.environ["ANALYST_AGENT_VERSION"]),
}
ANALYST_ENV_ID = os.environ["ANALYST_ENV_ID"]

# thread_ts -> session_id, so follow-ups land in the same session.
# Sessions stay open for replies. In production, persist this and
# archive sessions when threads go stale.
thread_sessions: dict[str, str] = {}

mrkdwn = SlackMarkdownConverter()

## 1. 봇이 멘션되면 세션 시작하기

Bolt는 모든 핸들러에 `ack` 콜백을 전달합니다. 이를 호출하면 이벤트를 받았다고 Slack에 알리는 것입니다. Slack은 [3초 안에](https://docs.slack.dev/apis/events-api/#responding) 확인되지 않은 이벤트를 재전송하므로, `on_mention`은 즉시 `ack()`를 호출하고 느린 작업(파일 업로드, 세션 생성, 스트리밍)은 백그라운드 스레드의 `start_analysis`에 넘깁니다.

멘션할 때마다 세션이 생성되며, [콘솔](https://platform.claude.com/)의 **Sessions**에서 열어 전체 추적을 볼 수 있습니다.

In [ ]:
@app.event("app_mention")
def on_mention(event, say, ack):
    ack()
    channel = event["channel"]
    thread_ts = event.get("thread_ts") or event["ts"]
    # Mention text arrives as "<@BOTID> question"; drop the mention prefix.
    question = event["text"].split(">", 1)[-1].strip()
    slack_file = (event.get("files") or [None])[0]

    say(text="On it. Analyzing now.", thread_ts=thread_ts)
    # Run the slow work in a background thread so this handler
    # returns within Slack's 3s limit.
    threading.Thread(target=start_analysis, args=(channel, thread_ts, question, slack_file)).start()


def start_analysis(channel: str, thread_ts: str, question: str, slack_file: dict | None) -> None:
    try:
        # If the mention had a file attached, pull it from Slack and
        # re-upload to the Anthropic Files API so the session can mount it.
        resources = []
        if slack_file:
            resp = requests.get(
                slack_file["url_private"],
                headers={"Authorization": f"Bearer {app.client.token}"},
                timeout=30,
            )
            resp.raise_for_status()
            mime = slack_file.get("mimetype", "text/csv")
            uploaded = client.beta.files.upload(
                file=(slack_file["name"], io.BytesIO(resp.content), mime)
            )
            mount = "/mnt/session/uploads/data.csv"
            resources.append({"type": "file", "file_id": uploaded.id, "mount_path": mount})
            question += f"\n\nThe data is mounted at {mount}."

        # One session per Slack thread. Store the thread coordinates in
        # metadata so anyone reading the event stream knows where to reply.
        session = client.beta.sessions.create(
            environment_id=ANALYST_ENV_ID,
            agent={"type": "agent", **ANALYST_AGENT},
            resources=resources,
            # Titles are capped at 80 chars and can't contain Unicode
            # control/format characters (Slack sometimes inserts them).
            title="".join(c for c in question if c.isprintable())[:80],
            metadata={"slack_channel": channel, "slack_thread_ts": thread_ts},
        )
        thread_sessions[thread_ts] = session.id

        # Send the question as a user.message event. The agent starts
        # working immediately; relay_stream posts its progress to the thread.
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
        )
        relay_stream(session.id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )

## 2. 진행 상황과 결과를 스레드로 중계하기

아래에 정의한 `relay_stream` 함수가 두 API를 잇는 다리입니다. Anthropic 세션 이벤트 스트림을 읽어 Slack에 게시합니다. 에이전트가 유휴 상태가 될 때까지 반복하다가, 최종 요약을 게시하고 에이전트가 작성한 파일을 업로드합니다.

`files.list(scope_id=...)`는 세션의 모든 파일을 반환합니다. 우리가 업로드한 CSV와 에이전트가 작성한 것 모두입니다. `downloadable == True`로 걸러서 사용자가 올린 입력이 아니라 에이전트가 만든 산출물(보고서, 차트)만 Slack에 게시되도록 합니다.

In [ ]:
def relay_stream(session_id: str, channel: str, thread_ts: str) -> None:
    summary = ""
    posted_progress = False
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            # Keep the latest text block; it becomes the final summary.
            for b in ev.content:
                if b.type == "text" and b.text.strip():
                    summary = b.text
        elif t == "agent.tool_use" and not posted_progress:
            # Post a one-time progress update when the agent starts
            # running commands.
            app.client.chat_postMessage(
                channel=channel, thread_ts=thread_ts, text="Running analysis..."
            )
            posted_progress = True
        elif t == "session.status_idle":
            break
        elif t == "session.status_terminated":
            trace = f"https://platform.claude.com/sessions/{session_id}"
            app.client.chat_postMessage(
                channel=channel,
                thread_ts=thread_ts,
                text=f"Session terminated unexpectedly. Trace: {trace}",
            )
            return

    # Turn is done. Post the summary, then upload any generated files.
    if summary:
        text = mrkdwn.convert(summary)
        if len(text) > 3900:  # Slack text limit ~4000 chars
            text = text[:3900] + "\n_(truncated)_"
        app.client.chat_postMessage(channel=channel, thread_ts=thread_ts, text=text)
    outputs = client.beta.files.list(scope_id=session_id, betas=["managed-agents-2026-04-01"])
    for f in outputs.data:
        if not f.downloadable:
            continue
        content = client.beta.files.download(f.id).read()
        app.client.files_upload_v2(
            channel=channel, thread_ts=thread_ts, filename=f.filename, content=content
        )

## 3. 같은 세션에서 후속 대화 처리하기

스레드의 답글은 기존 세션의 또 다른 턴이 됩니다. 봇을 다시 `@멘션`할 필요가 없습니다. 컨테이너 파일 시스템과 대화 기록이 턴을 넘어 유지됩니다.

In [ ]:
def continue_session(session_id: str, channel: str, thread_ts: str, text: str) -> None:
    try:
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )
        relay_stream(session_id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )


@app.event("message")
def on_thread_reply(event, ack):
    ack()
    thread_ts = event.get("thread_ts")
    # Only handle human replies in a thread where we already started
    # a session. Skip edits/deletes and other message subtypes.
    if event.get("subtype"):
        return
    if not thread_ts or event.get("bot_id") or thread_ts not in thread_sessions:
        return
    threading.Thread(
        target=continue_session,
        args=(thread_sessions[thread_ts], event["channel"], thread_ts, event["text"]),
    ).start()

## 4. 봇 실행하기

아래 셀은 Slack에 연결해 수신 대기를 시작합니다. 봇이 도는 동안 블로킹되므로, 끝내려면 ■ 중단 버튼을 누르세요.

봇이 들어와 있는 채널에서 CSV를 첨부해 멘션하세요. 진행 상황에 이어 요약과 `report.html`을 스레드에 게시합니다:

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/slack_data_bot/slack_thread.png" alt="봇의 분석 결과가 표시된 Slack 스레드" width="600" />

같은 데이터를 더 깊이 파고들려면 스레드에 답글을 다세요.

In [ ]:
SocketModeHandler(app, os.environ["SLACK_APP_TOKEN"]).start()

## 다음 단계

분석가 에이전트를 Slack 봇으로 감쌌습니다. 멘션이 세션을 시작하고, 이벤트 스트림이 진행 상황을 스레드로 중계하고, 산출물이 업로드되고, 답글이 같은 대화를 이어 갑니다.

- 분석 스타일을 바꾸려면 [`data_analyst_agent.ipynb`](data_analyst_agent.ipynb)에서 에이전트의 시스템 프롬프트를 바꾸세요. 그 노트북을 다시 실행하면 새 에이전트가 만들어지고 그 ID가 `.env`에 저장되어 봇이 사용합니다.
- 봇을 재시작해도 대화가 유지되도록 `thread_sessions`를 데이터베이스에 저장하세요.
- 봇을 이 노트북 밖으로 옮기세요. 코드를 `.py` 파일로 복사해, 오래 유지되는 WebSocket 연결을 감당할 수 있는 곳이면 어디든 배포할 수 있습니다.